# King County Housing Exploratory Data Analysis

Data source: King County house sales (2014–2015), Kaggle CC0. This notebook audits the source data and prepares the versioned datasets used by the models and Tableau workbook. Run it from the repository root.


In [ ]:
from pathlib import Path
import pandas as pd
from src.prepare_data import prepare, split_sales
from src.preprocessing import load_housing_data, select_features, build_preprocessor

DATA = Path("data/raw/kc_house_data.csv")
assert DATA.exists(), "First run: python -m src.download_kc"
df = load_housing_data(DATA)
print("Rows, columns:", df.shape)
display(df.head())


## Missing values and data structure
Repeated property IDs may represent resales. A zero renovation year records no renovation rather than a missing value.


In [ ]:
display(df.dtypes.to_frame("data_type"))
display(df.isna().sum().to_frame("missing_count"))
print("Identical duplicate rows:", df.duplicated().sum())
print("Additional rows with repeated IDs:", len(df) - df["id"].nunique())


## Distribution, correlations, and unusual values
The IQR rule is an exploratory flag, not an automatic deletion rule.


In [ ]:
from src.prepare_data import iqr_outlier_mask

display(df[["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms", "grade"]].describe())
print("Price outlier flags:", int(iqr_outlier_mask(df["price"]).sum()))
print("Living area outlier flags:", int(iqr_outlier_mask(df["sqft_living"]).sum()))
print("Bedrooms over 10:", int((df["bedrooms"] > 10).sum()))
print("Zero bathrooms:", int((df["bathrooms"] == 0).sum()))
display(df[["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms", "grade", "condition", "lat"]].corr().round(3))


## Train/test split and preprocessing
The property-level 80/20 split happens **before** imputation, encoding, scaling, and calculation of the classification cutoff.


In [ ]:
train, test = split_sales(df)
print("Train:", train.shape, "Test:", test.shape)
assert not set(train["id"]) & set(test["id"])
preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(select_features(train))
X_test = preprocessor.transform(select_features(test))
cutoff = (train["price"] / train["sqft_living"]).median()
print("Prepared X shapes:", X_train.shape, X_test.shape)
print("Training-only price/sqft classification cutoff:", round(cutoff, 2))


## Export prepared datasets
Open `tableau/house_sales_tableau.csv` in Tableau to create the charts described in `docs/TABLEAU_DASHBOARD.md`.


In [ ]:
report = prepare(DATA)
display(pd.Series(report, name="value").to_frame())
print("Tableau files:")
for path in sorted(Path("tableau").glob("*")):
    print(" -", path)
